# Analysis for the Prisoner's Dilemma with Punishment

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
w = 0.3
mu = 0.1
N = 500

## Custom functions

In [ ]:
# Computes fitness for A and B in a population of i A players and N-i B players. a and b refers to 
# the payoff strategy A earns against itself and against B respectively. c and d refers to
# the payoff B earns against A and B respectively

def f(a,b,w,N,i):
    return 1-w+w*((a*(i-1)+b*(N-i))/(N-1))

def g(c,d,w,N,i):
    return 1-w+w*((c*i+d*(N-i-1))/(N-1))

In [ ]:
def fixation(a, b, c, d, w, N):
    den = 1
    for k in range(1, N):
        gamma = 1
        for i in range(1, k + 1):
            print(str(g(c, d, w, N, i)))
            print(str(f(a, b, w, N, i)))
            gamma *= g(c, d, w, N, i) / f(a, b, w, N, i)
        den += gamma
    return 1 / den

In [ ]:
# Strategies considered (DPN inserted after CPN)
strategies = ["DNN", "CNN", "CPN", "DPN", "FNN", "FPN"]
n = len(strategies)

# Payoff matrix for the six strategies (order above)
payoff_matrix = np.array([
    [ 1, 3, 1, -1, 1, -1],  # DNN
    [ 0, 2, 2,  0, 0,  0],  # CNN
    [-1, 2, 2, -1, 2,  2],  # CPN
    [ 0, 3, 1, -2, 3,  1],  # DPN
    [ 1, 3, 2,  0, 1,  0],  # FNN
    [ 0, 3, 2, -1, 3,  2],  # FPN
])



# Matrix with pairwise fixation probability
fixation_matrix = np.zeros((len(strategies), len(strategies)))
for i in range(len(strategies)):
    for j in range(len(strategies)):
        if i != j:
            a = payoff_matrix[i][i]
            b = payoff_matrix[i][j]
            c = payoff_matrix[j][i]
            d = payoff_matrix[j][j]
            fixation_matrix[i][j] = fixation(a, b, c, d, w, N)


fixation_df = pd.DataFrame(fixation_matrix, index=strategies, columns=strategies)
print("Fixation Probability Matrix:")
print(fixation_df)

In [ ]:
fixation_df

In [ ]:
fixation_df.to_latex("fixation_matrix.tex",float_format="%.6f")

In [ ]:
# Transition matrix
T = pd.DataFrame(0.0, index=strategies, columns=strategies)

# Off-diagonal entries
for j in strategies:         
    for i in strategies:     
        if i != j:
            T.loc[j, i] = (mu/(n - 1)) * fixation_df.loc[i, j]

# Diagonal entries
for j in strategies:
    T.loc[j, j] = 1 - T.loc[j].drop(j).sum()

# Display or return T>
print("Transition matrix under rare mutations:")
print(T)

In [ ]:
T

In [ ]:
T.to_latex("transition_matrix.tex", float_format="%.6f")

## Stationary distribution

In [ ]:
# Compute eigenvalues and eigenvectors
S, U = np.linalg.eig(T.T)

# Extract stationary distribution
stationary = (U[:, np.isclose(S, 1)][:, 0] / U[:, np.isclose(S, 1)][:, 0].sum()).real

# Convert to DataFrame
stationary_df = pd.DataFrame(stationary, index=strategies, columns=["Stationary Dist."])

print(stationary_df)

In [ ]:
# Save to LaTeX
with open("stationary_distribution.tex", "w") as f:
    f.write(stationary_df.to_latex(float_format="%.6f"))

## Travelling time

In [ ]:
# Initial state: all DNN (index 0)
state = np.zeros(5)
state[0] = 1

# Track time steps until we're near certainty of being in FPN
threshold = 0.999  # You can adjust this as needed
max_steps = 1000
for t in range(1, max_steps + 1):
    state = state @ T
    if state[4] >= threshold:  # FPN is index 4
        print(f"Reached FPN with probability ≥ {threshold} at time step {t}")
        break
else:
    print("Did not reach the threshold within the max allowed steps.")
